## Wstęp do Uczenia Maszynowego

#### Laboratorium 03


### Zakres laboratorium

- macierz pomyłek
- krzywa ROC oraz AUC
- krzywa Precision–Recall i average precision
- przycinanie drzew klasyfikacyjnych
- walidacja krzyżowa i jej warianty
- dobór hiperparametrów


In [ ]:
# Import potrzebnych bibliotek i modułów
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.metrics import (
    RocCurveDisplay,
    confusion_matrix,
    accuracy_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    cross_val_score,
    train_test_split,
)


In [ ]:
# Wczytanie danych
pima = pd.read_csv("../lab02/pima.csv")


In [ ]:
# Definicja zbioru X i y
y = pima.Outcome
X = pima.drop(["Outcome"], axis = 1)

-----
#### 1. Macierz pomyłek i metryki klasyfikacji

W zadaniu klasyfikacji nie wystarczy wiedzieć, ile predykcji było poprawnych. Warto też sprawdzić, jakiego rodzaju błędy popełnia model. Macierz pomyłek zestawia klasy rzeczywiste z klasami przewidzianymi przez model.

- **True Positive (TP):** model przewidział klasę pozytywną i była to prawda;
- **True Negative (TN):** model przewidział klasę negatywną i była to prawda;
- **False Positive (FP):** model przewidział klasę pozytywną, ale była to pomyłka ("fałszywy alarm");
- **False Negative (FN):** model przewidział klasę negatywną, ale była to pomyłka ("przeoczenie")

<img src="confusion_matrix.png" width="400"/>

Na podstawie macierzy pomyłek obliczamy między innymi:

- **Dokładność (accuracy):** odsetek wszystkich poprawnych predykcji,
  $$\mathrm{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}.$$
- **Czułość (recall):** odsetek rzeczywistych pozytywów poprawnie rozpoznanych przez model,
  $$\mathrm{Recall} = \frac{TP}{TP + FN}.$$
- **Precyzję (precision):** odsetek trafnych predykcji pozytywnych,
  $$\mathrm{Precision} = \frac{TP}{TP + FP}.$$
- **Miara F1:** średnia harmoniczna precyzji i czułości,
  $$\mathrm{F1} = 2 \cdot \frac{\mathrm{Precision} \cdot \mathrm{Recall}}{\mathrm{Precision} + \mathrm{Recall}}.$$



---
##### *Zadanie 1*

Dla danych `pima` przygotuj podział na zbiory treningowy i testowy oraz dopasuj pełne drzewo klasyfikacyjne. Zapisz model w zmiennej `Tree`.

-----
##### *Zadanie 2*

Wyznacz macierz pomyłek oraz miary `accuracy`, `precision`, `recall` i `f1` dla predykcji drzewa klasyfikacyjnego z Zadania 1. Porównaj macierz pomyłek dla zbioru testowego z macierzą dla zbioru treningowego.


### 2. Krzywa ROC i AUC


Krzywa ROC (*Receiver Operating Characteristic*) służy do oceny binarnego klasyfikatora, który zwraca **wynik lub prawdopodobieństwo** klasy pozytywnej. Nie budujemy jej z gotowych etykiet 0/1, lecz zmieniamy próg odcięcia $t$:

$$\hat{y}=1 \quad \text{gdy} \quad \hat{p}(Y=1\mid X) \geq t.$$

Dla przypomnienia:

$$FPR = \frac{FP}{FP + TN}, \qquad TPR = \frac{TP}{TP + FN}.$$

Krzywa ROC przedstawia pary $(FPR, TPR)$ dla kolejnych progów: zaczynamy od progu większego niż wszystkie wyniki, czyli punktu $(0,0)$, a następnie obniżamy próg po kolejnych unikalnych wartościach wyniku. Przekątna odpowiada klasyfikatorowi losowemu.

Dla przykładu $y=[0,0,1,1]$ i wyników $[0.1,0.4,0.35,0.8]$ kolejne punkty ROC to $(0,0)$, $(0,0.5)$, $(0.5,0.5)$, $(0.5,1)$ i $(1,1)$.

**AUC** jest polem pod krzywą ROC. Wartość 0.5 odpowiada klasyfikatorowi losowemu, a 1 — idealnemu rozróżnieniu klas.

In [ ]:
# Przykład obliczenia punktów krzywej ROC
# Etykieta 1 oznacza klasę pozytywną, a scores_example — wynik modelu dla tej klasy.
y_example = np.array([0, 0, 1, 1])
scores_example = np.array([0.1, 0.4, 0.35, 0.8])
fpr_example, tpr_example, thresholds_example = roc_curve(y_example, scores_example)

pd.DataFrame({
    "próg": thresholds_example,
    "FPR": fpr_example,
    "TPR": tpr_example,
})

plt.plot(fpr_example, tpr_example, marker="o", label=f"AUC = {roc_auc_score(y_example, scores_example):.2f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="klasyfikator losowy")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Krzywa ROC dla małego przykładu")
plt.legend()
plt.show()


---
##### *Zadanie 3*

Dopasuj ustalone drzewo decyzyjne (`max_depth=3`) do zbioru treningowego. Na zbiorze testowym wyznacz prawdopodobieństwa klasy pozytywnej, narysuj krzywą ROC i oblicz AUC.

To ćwiczenie służy wyłącznie ilustracji ROC i AUC — nie wybieraj na podstawie tego wyniku hiperparametrów modelu.

### 3. Krzywa Precision–Recall i average precision

Krzywa Precision–Recall (PR) pokazuje, jak wraz ze zmianą progu zmieniają się precyzja i czułość:

$$\mathrm{Precision} = \frac{TP}{TP + FP}, \qquad \mathrm{Recall} = \frac{TP}{TP + FN}.$$

Jest szczególnie użyteczna przy klasie pozytywnej występującej rzadko. Punktem odniesienia dla klasyfikatora losowego jest wówczas udział klasy pozytywnej w danych. 

**Średnia precyzja (average precision, AP)** podsumowuje krzywą precision–recall jako średnią precyzji ważoną przyrostami czułości przy kolejnych progach decyzyjnych:

$$
\mathrm{AP} = \sum_n (R_n - R_{n-1}) P_n,
$$

gdzie $P_n$ i $R_n$ oznaczają odpowiednio precyzję i czułość dla $n$-tego progu. Każda para $(P_n, R_n)$ jest punktem krzywej PR.


---
##### *Zadanie 4*

Korzystając z prawdopodobieństw `y_score` wyznaczonych w Zadaniu 3, narysuj krzywą Precision–Recall i oblicz average precision (AP). Zaznacz na wykresie udział klasy pozytywnej w zbiorze testowym jako punkt odniesienia.


### 4. Przycinanie drzewa klasyfikacyjnego

Domyślne drzewo rośnie tak długo, jak pozwalają na to warunki zatrzymania algorytmu. Może więc być bardzo złożone i zbyt dokładnie odtwarzać przypadkowe cechy danych treningowych, zamiast wychwytywać zależności przydatne dla nowych obserwacji.

Przycinanie metodą *cost-complexity pruning* tworzy sekwencję coraz prostszych drzew. Parametr `ccp_alpha` jest karą za złożoność: dla `ccp_alpha = 0` otrzymujemy drzewo nieprzycięte, a większe wartości usuwają poddrzewa, które nie dają wystarczającego spadku nieczystości. Poniższe wykresy pokazują tę ścieżkę — zmianę nieczystości oraz liczby węzłów, liści i głębokości. Ostatnia wartość `ccp_alpha` prowadzi do drzewa z samym korzeniem, dlatego ją pomijamy.

Na tym etapie tylko obserwujemy zmianę struktury drzewa. Dobór `ccp_alpha` na podstawie walidacji krzyżowej wykonamy w dalszej części laboratorium.


In [ ]:
# Ścieżka przycinania dla pełnego drzewa z Zadania 1.
path = Tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

# Ostatnie ccp_alpha prowadzi do drzewa składającego się tylko z korzenia.
ccp_alphas_plot = ccp_alphas[:-1]
impurities_plot = impurities[:-1]

clfs = []
for alpha in ccp_alphas_plot:
    clf = tree.DecisionTreeClassifier(random_state=123, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    clfs.append(clf)


In [ ]:
# Zmiana całkowitej nieczystości liści wzdłuż ścieżki przycinania.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ccp_alphas_plot, impurities_plot, marker="o", drawstyle="steps-post")
ax.set_xlabel("ccp_alpha")
ax.set_ylabel("suma nieczystości liści")
ax.set_title("Ścieżka przycinania drzewa")
plt.show()


In [ ]:
# Wraz ze wzrostem ccp_alpha drzewo ma mniej węzłów i liści oraz jest płytsze.
node_counts = [clf.tree_.node_count for clf in clfs]
leaf_counts = [clf.get_n_leaves() for clf in clfs]
depths = [clf.tree_.max_depth for clf in clfs]

fig, ax = plt.subplots(3, 1, figsize=(7, 9), sharex=True)
ax[0].plot(ccp_alphas_plot, node_counts, marker="o", drawstyle="steps-post")
ax[0].set_ylabel("liczba węzłów")
ax[0].set_title("Złożoność drzewa a ccp_alpha")
ax[1].plot(ccp_alphas_plot, leaf_counts, marker="o", drawstyle="steps-post")
ax[1].set_ylabel("liczba liści")
ax[2].plot(ccp_alphas_plot, depths, marker="o", drawstyle="steps-post")
ax[2].set_xlabel("ccp_alpha")
ax[2].set_ylabel("głębokość drzewa")
fig.tight_layout()
plt.show()


### 5. Kroswalidacja (*ang. cross-validation*) vs Leave-One-Out


![](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png)

https://scikit-learn.org/stable/modules/cross_validation.html

---
##### *Zadanie 5*

Wymień zalety i wady kroswalidacji.


Zalety:


Wady:


---
##### *Zadanie 6*

Korzystając ze zbioru treningowego z Zadania 1, oszacuj jakość drzewa za pomocą 10-krotnej walidacji krzyżowej z miarą `accuracy`. Podaj średnią i odchylenie standardowe wyniku.


---
##### *Zadanie 7*

Na zbiorze treningowym wykonaj 100 powtórzeń 10-krotnej walidacji krzyżowej dla drzewa, używając miary `roc_auc`. Podaj średnią, odchylenie standardowe i narysuj rozkład wyników.

---
##### *Zadanie 8*

Dla danych `pima` zbadaj zachowanie hiperparametru modelu drzewa klasyfikacyjnego `max_leaf_nodes` w zakresie od 2 do 30. Na danych treningowych i w 5-krotnej CV, oblicz wartości następujących metryk: `accuracy`, `precision` i `roc_auc`. Przygotuj wizualizacje, wybierz wartość hiperparametru według `roc_auc` z CV, a następnie raz oceń wybrany model na zbiorze testowym.

> **Uwaga.** Hiperparametr wybieramy wyłącznie na podstawie wyniku walidacji krzyżowej. Poniżej oceniamy wybrany model na zbiorze testowym; wyniki testowe z wcześniejszych zadań służyły ilustracji metryk, a nie doborowi `max_leaf_nodes`.


### 6. Podsumowanie, czyli co warto wiedzieć...

1. Krzywa ROC i AUC oceniają zdolność modelu do rangowania obserwacji dla różnych progów.
2. Krzywa Precision–Recall oraz AP są szczególnie pomocne, gdy klasa pozytywna jest rzadka.
3. Przycinanie upraszcza drzewo; parametr `ccp_alpha` dobieramy na danych treningowych przez walidację krzyżową.
4. Walidacja krzyżowa służy do oceny modeli na zbiorze treningowym i pozwala oszacować zmienność wyniku.
5. Hiperparametry wybieramy według wyniku CV, a wybrany model oceniamy na zbiorze testowym.
